# Usando RAG local para análise de um artigo científico

- Cria chunks dos conteúdos extraídos

- Gera embeddings com modelo local 
  
- Constrói base vetorial com FAISS

## Carrega bibliotecas

In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, split, size, col, length
from pyspark.sql.types import StringType

from pyspark.sql.functions import udf, explode
from pyspark.sql.types import ArrayType, StringType

from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import AutoTokenizer, AutoModel
import torch

import faiss

import numpy as np
from tqdm import tqdm

## Define constantes

In [ ]:
INPUT_DOCS_PARSED_PATH = r'../data/interim/article_parsed_infos'

GENERATE_CHUNKS = True

MODEL_PATH = r'/home/Downloads/hf_models'

MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'

## Inicia sessão Spark

In [ ]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 15:11:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_parsed_docs = spark.read.parquet(INPUT_DOCS_PARSED_PATH)
df_parsed_docs.show()

+--------------------+------+--------------------+-----------+------------------+------------------+
|    modificationTime|length|          parsed_doc|  file_name|n_chars_parsed_doc|n_words_parsed_doc|
+--------------------+------+--------------------+-----------+------------------+------------------+
|2026-04-06 13:44:...| 47298|a strategic princ...|pg_0006.pdf|              4151|               608|
|2026-04-06 13:44:...| 34563|platform chooses ...|pg_0026.pdf|              3712|               540|
|2026-04-06 13:44:...| 34171|component. Given ...|pg_0004.pdf|              3823|               568|
|2026-04-06 13:44:...|169334|Divisiveness of C...|pg_0016.pdf|              3414|               562|
|2026-04-06 13:44:...|166820|Proof of Proposit...|pg_0042.pdf|              3708|               611|
|2026-04-06 13:44:...|159512|more likely to be...|pg_0023.pdf|              3556|               539|
|2026-04-06 13:44:...|170525|Reputability and ...|pg_0012.pdf|              3641|          

## Cria chunks

In [ ]:
if GENERATE_CHUNKS:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=70
    )

    def split_text_callable(text):
        if text is None:
            return []
        return text_splitter.split_text(text)

    split_udf = udf(split_text_callable, ArrayType(StringType()))

    df_chunks = df_parsed_docs.withColumn(
        'chunks',
        split_udf(df_parsed_docs['parsed_doc'])
    )

    df_chunks = df_chunks.withColumn('chunk', explode('chunks')).drop('chunks')

    df_chunks = df_chunks.withColumn(
        'n_chars_chunk',
        length(col('parsed_doc'))
    ).withColumn(
        'n_words_parsed_doc',
        size(split(col('parsed_doc'), r'\s+'))
    )

    print(df_chunks.show(5, truncate=100))

    df_chunks.write.mode('overwrite').parquet('../data/interim/chunks_article_parsed_infos')

else:
    df_chunks = spark.read.parquet('../data/interim/chunks_article_parsed_infos')
    print(df_chunks.show(5, truncate=100))

+-----------------------+------+----------------------------------------------------------------------------------------------------+-----------+------------------+------------------+----------------------------------------------------------------------------------------------------+-------------+
|       modificationTime|length|                                                                                          parsed_doc|  file_name|n_chars_parsed_doc|n_words_parsed_doc|                                                                                               chunk|n_chars_chunk|
+-----------------------+------+----------------------------------------------------------------------------------------------------+-----------+------------------+------------------+----------------------------------------------------------------------------------------------------+-------------+
|2026-04-06 13:44:18.356| 47298|a strategic principal who wants to persuade agents of an incorrect beli

None

In [6]:
print(f'#df_parsed_docs: {df_parsed_docs.count()} linhas ==> #df_chunks: {df_chunks.count()} linhas')

#df_parsed_docs: 42 linhas ==> #df_chunks: 338 linhas

## Gera embeddings

- `Qwen/Qwen3-Embedding-0.6B`: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/tree/main

In [7]:
tokenizer = AutoTokenizer.from_pretrained(f'{MODEL_PATH}/{MODEL_NAME}', trust_remote_code=True)
model = AutoModel.from_pretrained(f'{MODEL_PATH}/{MODEL_NAME}', trust_remote_code=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device, dtype=torch.float32)
model.eval()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [8]:
pd_chunks = df_chunks.select('chunk').toPandas()

In [9]:
def embed_texts(texts, batch_size=32, max_length=512):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
texts = pd_chunks['chunk'].tolist()

embeddings = embed_texts(
    texts,
    batch_size=16
)

100%|██████████| 85/85 [00:34<00:00,  2.44it/s]


In [11]:
embeddings.shape

(338, 1024)

In [12]:
faiss.normalize_L2(embeddings)  # v <-- v/∣v∣

In [ ]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f'index.ntotal: {index.ntotal}')

index.ntotal: 338

## Exporta como FAISS

In [14]:
faiss.write_index(index, '../data/processed/chunks.index')

pd_chunks['chunk_emb'] = embeddings.tolist()
pd_chunks.to_parquet('../data/processed/chunks_with_embeddings.parquet')

### Testa busca semântica

In [15]:
def search(query, k=5):
    q_emb = embed_texts([query], batch_size=1)
    faiss.normalize_L2(q_emb)

    D, I = index.search(q_emb, k)
    return pd_chunks.iloc[I[0]]

In [16]:
results = search('misinformation in social media')

print(results['chunk'].tolist())

100%|██████████| 1/1 [00:00<00:00,  2.95it/s]


[
    'misinformation is less likely (Altay et al. (2020)), and when such reputational concerns are missing, even 
calling out individuals sharing misinformation is fairly ineffective (Mosleh et al. (2021 a)). Finally, this 
proposition provides a possible pathway for low-reliability content (often containing misinformation) to become 
viral. Vosoughi et al. (2018) argued that misinformation spreads farther, faster, deeper and more broadly than 
truthful news on social media',
    '. Pennycook and Rand (2019 a) document that social media users recognize low-reliability content sources (e. 
g., Breitbart or Infowars), and this content is not typically shared by attentive social media users, regardless of
partisanship (see Pennycook and Rand (2019 b)). This proposition also clariﬁes that viral spread of misinformation 
is not a mechanical effect in our model: if anything, less reliable articles that are more likely to contain 
misinformation are less likely to become viral',
    '...”.19 6 Regulation Our analysis so far raises the natural question of what types of regulations might 
counter the viral spread of misinformation and platform choices leading to excessive ideological homophily. We 19 
See Vanity Fair: https://www. vanityfair. 
com/news/2020/12/with-the-election-over-facebook-gets-back-to-spreading-misinformation and also https://www. 
technologyreview. com/2021/03/11/1020600/facebook-responsible-ai-misinformation/. 17',
    '. This result highlights an important channel by which misinformation spreads: it is precisely when articles 
are likely to contain misinformation that the platform seeks to maximize engagement by creating (endogenous) echo 
chambers, or ﬁlter bubbles, where these articles spread virally within like-minded communities. Put differently, 
with low reliability content, neither the platform nor the users are disciplined about sharing misinformation, and 
so these news items spread virtually uninhibited',
    '7 Conclusion This paper has developed a simple model of the spread of misinformation over social media 
platforms. A group of Bayesian agents with heterogeneous priors receive and share news items (articles) according 
to a stochastic sharing network, determined by the social media platform. Articles may be truthful and informative 
about an underlying state, or may contain misinformation, making them (weakly) anti-correlated with the underlying 
state'
]

## Interrompe sessão Spark

In [17]:
spark.stop()